# FunctionGemma LoRA Fine-Tuning on GPU & LiteRT-LM Deployment

<table class="tfo-notebook-buttons">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_1_LoRA_Fine_Tuning.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
</table>

[LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM) is Google’s high-performance edge inference engine that runs LLMs across mobile, web, and desktop with full CPU/GPU/NPU acceleration and production-ready SDKs.

This notebook fine-tunes [FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it) with a LoRA adapter using [Unsloth](https://github.com/unslothai/unsloth) QLoRA on a Colab GPU, converts it to `.litertlm` (`llm_model_type: function_gemma`) for on-device tool calling, and runs it on your phone with [Google AI Edge Gallery](https://developers.google.com/edge/gallery).

LiteRT-LM supports two ways to deploy a LoRA, and this notebook produces both:

| | What you ship | When to use it |
|---|---|---|
| **Dynamic LoRA** | A base `.litertlm` plus a small adapter `.tflite`, attached at runtime | Several adapters sharing one base model |
| **Static fusion** | One `.litertlm` with the adapter merged into the weights | A single task, or apps without runtime LoRA support (such as AI Edge Gallery) |

The companion notebook [FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb](https://colab.research.google.com/github/google-ai-edge/LiteRT-LM/blob/main/colabs/FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb) shows the dynamic path on its own, using the files this notebook publishes in Step 11.

**Setup**:
- Connect to a **T4 GPU** runtime (**Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**). The free tier is enough.
- **Estimated Runtime**: Expect **~30 minutes** end-to-end on a T4 GPU (more if you increase `MAX_STEPS` in Step 4 or `EVAL_N` in Step 10).
- Add `HF_TOKEN` in Colab **Secrets (🔑)** to download the gated `google/functiongemma-270m-it` repository (accept the [FunctionGemma license](https://huggingface.co/google/functiongemma-270m-it) first). The token needs **write** access if you upload the models in Step 11.
- For Step 12, install [Google AI Edge Gallery](https://developers.google.com/edge/gallery) on an Android or iOS phone.

## The example task

**Goal:** teach FunctionGemma a custom smart-home tool set. It has three tools, `set_device_state(device_id, state)`, `set_brightness(device_id, level)` and `set_temperature(zone_id, celsius)`. The targets use canonical IDs (`light.living_room`, `climate.office`) and numeric arguments that the base model has not been trained on.

The tools are passed to the model the same way LiteRT-LM passes them on device, with `engine.create_conversation(tools=...)`. These two held-out requests are used throughout the FunctionGemma Colabs:

| Request | Expected tool call |
|---|---|
| *Turn on the living room lights* | `set_device_state(device_id="light.living_room", state="on")` |
| *I want the office at 22 degrees* | `set_temperature(zone_id="climate.office", celsius=22)` |

**Before and after fine-tuning.** Recorded outputs from Step 10 (default settings, T4 GPU):

| Request | Base model | Fine-tuned (fused) |
|---|---|---|
| *Turn on the living room lights* | ❌ `set_device_state(device_id="living_room_lights", state="on")` | ✅ `set_device_state(device_id="light.living_room", state="on")` |
| *I want the office at 22 degrees* | ❌ no tool call | ✅ `set_temperature(zone_id="climate.office", celsius=22)` |
| Held-out requests (n=100): correct function / exact call | 74% / 0% | 100% / 96% |

**Training setup:**

* **Dataset:** about 420 synthetic smart-home requests (Step 3). Some devices, rooms (such as the living-room lights) and phrasings appear **only** in the held-out evaluation set. Prompts are rendered exactly as LiteRT-LM's `FunctionGemmaDataProcessor` renders them at inference (tool declarations in the `developer` turn), so training matches on-device inference.
* **Adapter:** rank 32, alpha 64, on the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the layers LiteRT-LM can attach a LoRA to at runtime.
* **Training:** `MAX_STEPS` (default 250) steps at effective batch size 8 (`per_device_train_batch_size=2`, `gradient_accumulation_steps=4`), cosine schedule with `lr=5e-4`. Loss is computed only on the function call after `<start_of_turn>model`.

## What you will do

**Fine-tune the model**

* **Step 1.** Install the training libraries.
* **Step 2.** Load FunctionGemma in 4-bit and attach LoRA adapters.
* **Step 3.** Build the smart-home tool-calling dataset.
* **Step 4.** Train the adapter.
* **Step 5.** Restart the runtime to get a clean Python process for conversion.

**Convert to LiteRT-LM and verify**

* **Step 6.** Install the conversion libraries.
* **Step 7.** Convert the base model to `.litertlm`.
* **Step 8.** Convert the adapter to `.tflite` (for dynamic LoRA).
* **Step 9.** Merge the adapter into the base model and convert it to a single `.litertlm` (static fusion).
* **Step 10.** Evaluate the base and fine-tuned models on held-out requests.

**Run it on your phone**

* **Step 11.** Publish the models to Hugging Face and get a QR code.
* **Step 12.** Run the model in the AI Edge Gallery app.

## Fine-tune the model

### Step 1: Install the training libraries

Installs Unsloth and the Hugging Face training stack (`trl`, `peft`, `bitsandbytes`, `datasets`).

In [ ]:
!pip install -q --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.18.2,<=0.24.0" peft accelerate bitsandbytes datasets "typing-extensions>=4.13.0"

### Step 2: Load FunctionGemma and attach LoRA adapters

Loads the base model in 4-bit to save GPU memory (QLoRA), then attaches rank-32 LoRA adapters to the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) in every transformer block. Only the adapter weights are trained; the base weights stay frozen.

In [ ]:
import os
import sys
import importlib
import typing_extensions

os.environ["WANDB_DISABLED"] = "true"
importlib.reload(typing_extensions)
if not hasattr(typing_extensions, "Sentinel"):
    typing_extensions.Sentinel = object()

# Clean up any partially initialized modules from earlier failed imports in the same kernel
for _mod in list(sys.modules.keys()):
    if _mod.startswith(("wandb", "pydantic", "unsloth")):
        sys.modules.pop(_mod, None)

import unsloth
from unsloth import FastLanguageModel
import torch

MODEL_ID = "google/functiongemma-270m-it"
MAX_SEQ_LENGTH = 2048
LORA_RANK = 32  # Upgraded rank for enhanced parameter capacity on FunctionGemma 270M
LORA_ALPHA = 64
ADAPTER_OUTPUT_DIR = "./functiongemma_270m_lora"

print(f"🔄 Loading {MODEL_ID} in 4-bit via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,
)

# Apply Attention-Projection Adaptation with Rank 32 / Alpha 64 (matching LiteRT-LM Dynamic LoRA slots)
print("🎯 Attaching LoRA adapters across attention projection layers (q_proj, k_proj, v_proj, o_proj)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
model.print_trainable_parameters()

### Step 3: Build the smart-home tool-calling dataset

Generates synthetic requests for the three smart-home tools and writes them to `/content/sft_data`:

- `train.jsonl`: each example is the prompt, rendered exactly as LiteRT-LM renders it on device, followed by the target call, e.g. `<start_function_call>call:set_device_state{device_id:<escape>light.kitchen<escape>,state:<escape>on<escape>}<end_function_call>`.
- `eval_toolcalls.jsonl`: held-out requests (unseen phrasings and devices) used in Step 10.
- `tools.json` and `gallery_system_prompt.txt`: the tool schema and the matching AI Edge Gallery system prompt (Step 12).

In [ ]:
# @title { display-mode: "form" }
import json
import os
import random

SFT_DATA_DIR = "/content/sft_data"
NUM_TRAIN = 1500
NUM_EVAL = 300
os.makedirs(SFT_DATA_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. Smart-home tools. The model must learn the canonical ID conventions
#    (`light.living_room`, `climate.bedroom`) and numeric arguments.
# ---------------------------------------------------------------------------
TOOLS = [
    {
        "name": "set_device_state",
        "description": "Turn a smart-home device on or off.",
        "parameters": {
            "type": "object",
            "properties": {
                "device_id": {"type": "string", "description": "Device ID formatted as <device_type>.<room>"},
                "state": {"type": "string", "description": "on or off"},
            },
            "required": ["device_id", "state"],
        },
    },
    {
        "name": "set_brightness",
        "description": "Set the brightness of a light.",
        "parameters": {
            "type": "object",
            "properties": {
                "device_id": {"type": "string", "description": "Light ID formatted as light.<room>"},
                "level": {"type": "integer", "description": "Brightness percentage from 0 to 100"},
            },
            "required": ["device_id", "level"],
        },
    },
    {
        "name": "set_temperature",
        "description": "Set the target temperature of a room.",
        "parameters": {
            "type": "object",
            "properties": {
                "zone_id": {"type": "string", "description": "Climate zone ID formatted as climate.<room>"},
                "celsius": {"type": "integer", "description": "Target temperature in degrees Celsius"},
            },
            "required": ["zone_id", "celsius"],
        },
    },
]
with open(os.path.join(SFT_DATA_DIR, "tools.json"), "w", encoding="utf-8") as f:
    json.dump(TOOLS, f, indent=2)


# ---------------------------------------------------------------------------
# 2. Render prompts exactly as LiteRT-LM does at inference time
#    (FunctionGemmaDataProcessor + FunctionGemma chat template), so training
#    and on-device inference see identical text. Each training example is the
#    rendered prompt followed by the target function call.
# ---------------------------------------------------------------------------
def fc_value(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return f"<escape>{value}<escape>"
    if isinstance(value, (int, float)):
        return json.dumps(value)
    if isinstance(value, list):
        return "[" + ",".join(fc_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{fc_value(v)}" for k, v in value.items()) + "}"
    raise TypeError(f"Unsupported value: {value!r}")


def uppercase_types(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if k == "type" and v in ("string", "number", "integer", "object", "array", "boolean", "null"):
                out[k] = v.upper()
            else:
                out[k] = uppercase_types(v)
        return out
    if isinstance(obj, list):
        return [uppercase_types(v) for v in obj]
    return obj


def fc_declaration(tool: dict) -> str:
    fields = {k: v for k, v in uppercase_types(tool).items() if k != "name"}
    return f"declaration:{tool['name']}{fc_value(fields)}"


DEVELOPER_TURN = (
    "<start_of_turn>developer\n"
    + "".join(f"<start_function_declaration>{fc_declaration(t)}<end_function_declaration>" for t in TOOLS)
    + "<end_of_turn>\n"
)


def render_prompt(query: str) -> str:
    return DEVELOPER_TURN + f"<start_of_turn>user\n{query.strip()}<end_of_turn>\n<start_of_turn>model\n"


def render_call(name: str, args: dict) -> str:
    arg_str = ",".join(f"{k}:{fc_value(v)}" for k, v in args.items())
    return f"<start_function_call>call:{name}{{{arg_str}}}<end_function_call><start_function_response>"


# ---------------------------------------------------------------------------
# 3. Synthetic smart-home requests with held-out rooms/devices and phrasings.
# ---------------------------------------------------------------------------
ROOMS = {
    "living_room": ["living room", "lounge"], "kitchen": ["kitchen"], "bedroom": ["bedroom"],
    "office": ["office", "study"], "garage": ["garage"], "bathroom": ["bathroom"],
    "dining_room": ["dining room"], "hallway": ["hallway", "hall"], "basement": ["basement"],
}
DEVICES = {"light": ["lights", "light", "lamps"], "fan": ["fan", "ceiling fan"], "tv": ["TV", "television"], "plug": ["smart plug", "outlet"]}
HELD_OUT_PAIRS = {("light", "living_room"), ("fan", "bedroom"), ("tv", "office"), ("light", "garage")}
HELD_OUT_CLIMATE = {"living_room", "office"}

STATE_TEMPLATES = {
    "train": {
        "on": ["Turn on the {room} {dev}", "Switch on the {dev} in the {room}", "Please power up the {room} {dev}", "Can you turn the {room} {dev} on?"],
        "off": ["Turn off the {room} {dev}", "Switch off the {dev} in the {room}", "Shut off the {dev} in the {room}", "Please kill the {room} {dev}"],
    },
    "eval": {
        "on": ["I'd like the {room} {dev} on", "Activate the {dev} in the {room}"],
        "off": ["Could you turn the {room} {dev} off", "Deactivate the {dev} in the {room}"],
    },
}
BRIGHTNESS_TEMPLATES = {
    "train": ["Set the {room} {dev} to {pct} percent", "Dim the {room} {dev} to {pct}%", "Change the {dev} in the {room} to {pct} percent brightness"],
    "eval": ["Put the {dev} in the {room} at {pct}%", "I want the {room} {dev} at {pct} percent"],
}
TEMP_TEMPLATES = {
    "train": ["Set the {room} to {t} degrees", "Make it {t} degrees in the {room}", "Heat the {room} to {t} degrees Celsius"],
    "eval": ["I want the {room} at {t} degrees", "Change the {room} temperature to {t}"],
}

rng = random.Random(0)
train_rows, eval_rows, seen_queries = [], [], set()


def add_example(bucket: list, query: str, name: str, args: dict) -> None:
    if query in seen_queries:
        return
    seen_queries.add(query)
    bucket.append({"query": query, "name": name, "arguments": args})


for dev_type, dev_names in DEVICES.items():
    for room_id, room_names in ROOMS.items():
        held_out = (dev_type, room_id) in HELD_OUT_PAIRS
        # Held-out devices only appear in eval, with both training and eval phrasings.
        plans = [("eval", "train"), ("eval", "eval")] if held_out else [("train", "train"), ("eval", "eval")]
        for target, phrasing in plans:
            bucket = train_rows if target == "train" else eval_rows
            for state, templates in STATE_TEMPLATES[phrasing].items():
                for tmpl in templates:
                    add_example(bucket, tmpl.format(room=rng.choice(room_names), dev=rng.choice(dev_names)),
                                "set_device_state", {"device_id": f"{dev_type}.{room_id}", "state": state})
            if dev_type == "light":
                for tmpl in BRIGHTNESS_TEMPLATES[phrasing]:
                    for pct in rng.sample(range(5, 100, 5), 4):
                        add_example(bucket, tmpl.format(room=rng.choice(room_names), dev=rng.choice(dev_names), pct=pct),
                                    "set_brightness", {"device_id": f"light.{room_id}", "level": pct})

for room_id, room_names in ROOMS.items():
    held_out = room_id in HELD_OUT_CLIMATE
    plans = [("eval", "train"), ("eval", "eval")] if held_out else [("train", "train"), ("eval", "eval")]
    for target, phrasing in plans:
        bucket = train_rows if target == "train" else eval_rows
        for tmpl in TEMP_TEMPLATES[phrasing]:
            for t in rng.sample(range(16, 29), 4):
                add_example(bucket, tmpl.format(room=rng.choice(room_names), t=t), "set_temperature", {"zone_id": f"climate.{room_id}", "celsius": t})

# Showcase prompts used in Step 10 -- held out from training (same prompts as the other FunctionGemma Colabs).
DEMO_EXAMPLES = [
    {"query": "Turn on the living room lights", "name": "set_device_state", "arguments": {"device_id": "light.living_room", "state": "on"}},
    {"query": "I want the office at 22 degrees", "name": "set_temperature", "arguments": {"zone_id": "climate.office", "celsius": 22}},
]
demo_queries = {d["query"] for d in DEMO_EXAMPLES}
train_rows = [r for r in train_rows if r["query"] not in demo_queries]
eval_rows = [r for r in eval_rows if r["query"] not in demo_queries]

rng.shuffle(train_rows)
rng.shuffle(eval_rows)
train_rows = train_rows[:NUM_TRAIN]
eval_rows = DEMO_EXAMPLES + eval_rows[: NUM_EVAL - len(DEMO_EXAMPLES)]
train_queries = {r["query"] for r in train_rows}
assert not any(r["query"] in train_queries for r in eval_rows), "Eval queries must not appear in training."

with open(os.path.join(SFT_DATA_DIR, "train.jsonl"), "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps({"text": render_prompt(r["query"]) + render_call(r["name"], r["arguments"])}) + "\n")
# AI Edge Gallery (Step 12) renders its system prompt as the developer turn, so pasting the
# tool declarations there reproduces the training prompt format on the phone.
with open(os.path.join(SFT_DATA_DIR, "gallery_system_prompt.txt"), "w", encoding="utf-8") as f:
    f.write(DEVELOPER_TURN[len("<start_of_turn>developer\n"):-len("<end_of_turn>\n")])
with open(os.path.join(SFT_DATA_DIR, "eval_toolcalls.jsonl"), "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps(r) + "\n")

print(f"[OK] Smart-home SFT splits: train={len(train_rows)}, held-out eval={len(eval_rows)} (written to {SFT_DATA_DIR})")
print("Sample training example:", train_rows[0]["query"], "->", render_call(train_rows[0]["name"], train_rows[0]["arguments"]))

### Step 4: Train the adapter

Trains the LoRA adapter for `MAX_STEPS` steps, computing the loss only on the function call that follows `<start_of_turn>model`, and saves it to `./functiongemma_270m_lora`.

In [ ]:
# @title { display-mode: "form" }
import torch
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling, TrainingArguments
from trl import SFTTrainer

MAX_STEPS = 250  # @param {type:"integer"}
SFT_DATA_DIR = "/content/sft_data"

# DataCollatorForCompletionOnlyLM moved between TRL releases; fall back to an equivalent collator.
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
            def __init__(self, response_template, tokenizer, **kwargs):
                super().__init__(tokenizer=tokenizer, mlm=False, **kwargs)
                self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)

            def torch_call(self, examples):
                batch = super().torch_call(examples)
                labels = batch["labels"].clone()
                resp_ids = self.response_token_ids
                for i in range(labels.size(0)):
                    seq = labels[i].tolist()
                    matched = False
                    for idx in range(len(seq) - len(resp_ids) + 1):
                        if seq[idx : idx + len(resp_ids)] == resp_ids:
                            labels[i][: idx + len(resp_ids)] = -100
                            matched = True
                            break
                    if not matched:
                        labels[i][:] = -100
                batch["labels"] = labels
                return batch

train_dataset = load_dataset("json", data_files=f"{SFT_DATA_DIR}/train.jsonl", split="train")
print(f"Loaded {len(train_dataset)} smart-home training examples.")

# Compute the loss only on the function call that follows `<start_of_turn>model\n`.
response_template = "<start_of_turn>model\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)
sample_batch = collator([{"input_ids": tokenizer.encode(train_dataset[0]["text"])}])
supervised_ids = [t.item() for t in sample_batch["labels"][0] if t.item() != -100]
assert supervised_ids, "Response template was not matched in tokenized example!"
print(f"Supervised target preview: {tokenizer.decode(supervised_ids)!r}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    data_collator=collator,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=5e-4,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        report_to="none",
        output_dir="./checkpoints",
    ),
)

print("🚀 Starting optimized LoRA fine-tuning for FunctionGemma (270M)...")
trainer.train()

model.save_pretrained(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)
print(f"✅ Trained FunctionGemma (270M) LoRA adapter saved to: {ADAPTER_OUTPUT_DIR}")

### Step 5: Restart the runtime

Unsloth patches `transformers` and `torch` in memory when it is imported, and Step 6 installs different versions of several of the same packages. This cell ends the Python process so the conversion steps start fresh with the newly installed packages.

> [!NOTE]
> Colab will say the session crashed. That is expected. Files on disk, including the trained adapter in `./functiongemma_270m_lora` and the dataset in `/content/sft_data`, are kept. Continue with Step 6; do not re-run Steps 1–4.

In [ ]:
import os
os._exit(0)  # Hard exit forces Colab to launch a completely clean Python process

## Convert to LiteRT-LM and verify

### Step 6: Install the conversion libraries

Installs `litert-torch` (the PyTorch-to-LiteRT converter), the `litert-lm` runtime, and the tools used in Steps 7–11.

In [ ]:
# @title { display-mode: "form" }
!pip install -q "protobuf>=5.26.1,<6.0.0dev"
!pip install -q --upgrade transformers litert-torch ai-edge-quantizer flatbuffers litert-lm litert-lm-builder litert-converter fire jaxtyping peft bitsandbytes huggingface_hub qrcode[pil]
!pip install -q --no-deps "typing-extensions>=4.13.0"

import os
import sys
import importlib
import typing_extensions
importlib.reload(typing_extensions)

# Bypass protobuf version validation errors across all protobuf runtime versions
try:
    import google.protobuf.runtime_version
    google.protobuf.runtime_version.ValidateProtobufRuntimeVersion = lambda *args, **kwargs: None
except Exception:
    pass

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
importlib.invalidate_caches()

# Verify protobuf version via google.protobuf module
import google.protobuf
print(f"✅ Protobuf version: {google.protobuf.__version__}")
print("✅ LiteRT-LM conversion dependencies successfully installed!")

### Step 7: Convert the base model to `.litertlm`

Exports `google/functiongemma-270m-it` to `/content/functiongemma_270m_litertlm/model.litertlm` with **`litert_lm_model_type_override="function_gemma"`**, so LiteRT-LM selects `FunctionGemmaDataProcessor` (`llm_model_type { function_gemma {} }`) and stops at `<start_function_response>`. The chat template is patched so it accepts the pre-formatted tool declarations LiteRT-LM passes to it. This is the baseline for Step 10. Log in with your Hugging Face token when prompted.

**Quantization Recipe Trade-offs (`QUANTIZATION_RECIPE`)**:
- **`dynamic_wi8_afp32` (8-bit dynamic weights, `float32` activations — Default)**: Higher precision (~285 MB `.litertlm`). Recommended for compact models like FunctionGemma 270M where strict tool-call syntax and argument accuracy are sensitive to weight quantization.
- **`dynamic_wi4b32_afp32` (4-bit blockwise weights with block size 32, `float32` activations)**: Smaller memory footprint (~170 MB `.litertlm`, ~40% smaller) and lower memory bandwidth on edge devices, trading a slight reduction in tool-call schema fidelity for smaller size.

Step 9 uses the same recipe for the fused model.

In [ ]:
# @title { display-mode: "form" }
import importlib
import os
import re

# Bypass protobuf version validation errors across all protobuf runtime versions
try:
    import google.protobuf.runtime_version
    google.protobuf.runtime_version.ValidateProtobufRuntimeVersion = lambda *args, **kwargs: None
except Exception:
    pass

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
importlib.invalidate_caches()

import jinja2
import torch
import transformers
from huggingface_hub import snapshot_download
from litert_torch.generative.export_hf import export as export_hf
from litert_torch.generative.export_hf.core import cache_base as _lt_cache_base
from litert_torch.generative.export_hf.core import litert_lm_builder as lmb_core
from litert_torch.generative.export_hf.model_ext import patches as patches_lib
import litert_torch.generative.export_hf.model_ext.gemma3.patch as _  # registers the Gemma3 patches

QUANTIZATION_RECIPE = "dynamic_wi8_afp32"  # @param ["dynamic_wi8_afp32", "dynamic_wi4b32_afp32"]
BASE_MODEL_ID = "google/functiongemma-270m-it"
BASE_MODEL_DIR = "/content/base_functiongemma_hf"
BASE_OUTPUT_DIR = "/content/functiongemma_270m_litertlm"
BASE_LITERTLM_PATH = os.path.join(BASE_OUTPUT_DIR, "model.litertlm")

# Newer litert-lm-builder releases removed `TfLiteModelType` (model types are now passed as strings such as
# "prefill_decode"); map the enum-style attributes litert-torch still uses onto those strings.
class _ModelTypeStr(str):
    @property
    def name(self):
        return self.upper()

    @property
    def value(self):
        return str(self)


class _TfLiteModelTypeCompat(type):
    def __getattr__(cls, attr):
        if attr.startswith("_"):
            raise AttributeError(attr)
        return _ModelTypeStr(attr.lower())


class _TfLiteModelTypeShim(metaclass=_TfLiteModelTypeCompat):
    pass


_builder_mods = [getattr(lmb_core, "litertlm_builder", None)]
try:
    import litert_lm_builder as _pip_builder
    _builder_mods.append(_pip_builder)
except ImportError:
    pass
for _bm in _builder_mods:
    if _bm is not None and not hasattr(_bm, "TfLiteModelType"):
        _bm.TfLiteModelType = _TfLiteModelTypeShim

# Cache API compatibility with the installed transformers release.
if not hasattr(_lt_cache_base.LiteRTLMCacheMixin, "get_max_length"):
    _lt_cache_base.LiteRTLMCacheMixin.get_max_length = lambda self, layer_idx=None: (
        -1 if (layer_idx is not None and layer_idx >= len(self.layers))
        else (
            max((getattr(l, "get_max_length", getattr(l, "get_max_cache_shape", lambda: -1))() for l in self.layers), default=-1)
            if layer_idx is None
            else getattr(self.layers[layer_idx], "get_max_length", getattr(self.layers[layer_idx], "get_max_cache_shape", lambda: -1))()
        )
    )
if not hasattr(_lt_cache_base.LiteRTLMCacheMixin, "get_max_cache_shape"):
    _lt_cache_base.LiteRTLMCacheMixin.get_max_cache_shape = lambda self, layer_idx=0: self.get_max_length(layer_idx)
if not hasattr(_lt_cache_base.LiteRTLMCacheMixin, "max_cache_len"):
    _lt_cache_base.LiteRTLMCacheMixin.max_cache_len = property(lambda self: self.get_max_length())

# Register function_gemma in the Gemma3 patch registries so litert_lm_model_type_override="function_gemma"
# applies the Gemma3 RMSNorm/MLP patches.
for _registry in (patches_lib._CONTEXT_REGISTRY, patches_lib._MODEL_PATCH_REGISTRY):
    if "gemma3_text" in _registry:
        _registry["function_gemma"] = _registry["gemma3_text"]

# Write `llm_model_type: function_gemma` and stop at `<start_function_response>` in the LlmMetadata.
if not getattr(lmb_core.build_llm_metadata, "_function_gemma_patched", False):
    _orig_build_llm_metadata = lmb_core.build_llm_metadata

    def _fg_build_llm_metadata(*args, **kwargs):
        llm_metadata = _orig_build_llm_metadata(*args, **kwargs)
        llm_metadata.llm_model_type.CopyFrom(
            lmb_core.llm_model_type_pb2.LlmModelType(function_gemma=lmb_core.llm_model_type_pb2.FunctionGemma())
        )
        existing_stops = {st.token_str for st in llm_metadata.stop_tokens if st.HasField("token_str")}
        for stop in ("<end_of_turn>", "<start_function_response>"):
            if stop not in existing_stops:
                llm_metadata.stop_tokens.add().token_str = stop
        return llm_metadata

    _fg_build_llm_metadata._function_gemma_patched = True
    lmb_core.build_llm_metadata = _fg_build_llm_metadata

# LiteRT-LM's FunctionGemmaDataProcessor passes tools to the chat template as pre-formatted
# "declaration:..." strings; let the template's format_function_declaration macro accept them.
FC_STR_START = "{%- if tool_data is string -%}{{- tool_data -}}{%- else -%}"
FC_STR_END = "{%- endif -%}{#- fc-string-tool -#}"


def _patch_template_text(tmpl: str) -> str:
    hdr = "{% macro format_function_declaration(tool_data) -%}"
    hdr_idx = tmpl.find(hdr)
    if hdr_idx == -1 or FC_STR_END in tmpl:
        return tmpl
    body_start = hdr_idx + len(hdr)
    end_match = re.compile(r"\{%-?\s*endmacro").search(tmpl, body_start)
    if not end_match:
        return tmpl
    body = tmpl[body_start:end_match.start()]
    if FC_STR_START not in body:
        body = body.replace("declaration:", FC_STR_START + "declaration:", 1)
    patched = tmpl[:body_start] + body + FC_STR_END + tmpl[end_match.start():]
    jinja2.Environment().parse(patched)
    return patched


def patch_fc_string_tools(model_dir: str) -> None:
    """Patches the chat template, whether it is stored in chat_template.jinja or tokenizer_config.json."""
    jinja_path = os.path.join(model_dir, "chat_template.jinja")
    if os.path.isfile(jinja_path):
        with open(jinja_path, "r", encoding="utf-8") as f:
            tmpl = f.read()
        patched = _patch_template_text(tmpl)
        if patched != tmpl:
            with open(jinja_path, "w", encoding="utf-8") as f:
                f.write(patched)
    cfg_path = os.path.join(model_dir, "tokenizer_config.json")
    if os.path.isfile(cfg_path):
        import json
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if isinstance(cfg.get("chat_template"), str):
            patched = _patch_template_text(cfg["chat_template"])
            if patched != cfg["chat_template"]:
                cfg["chat_template"] = patched
                with open(cfg_path, "w", encoding="utf-8") as f:
                    json.dump(cfg, f, indent=2, ensure_ascii=False)
    print(f"[OK] Chat template in {model_dir} accepts LiteRT-LM pre-formatted tool declarations")


hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    from huggingface_hub import notebook_login
    notebook_login()

if not os.path.isfile(os.path.join(BASE_MODEL_DIR, "config.json")):
    print(f"Downloading {BASE_MODEL_ID} to {BASE_MODEL_DIR} ...")
    snapshot_download(repo_id=BASE_MODEL_ID, local_dir=BASE_MODEL_DIR, token=hf_token)
patch_fc_string_tools(BASE_MODEL_DIR)

print(f"Exporting base FunctionGemma 270M ({QUANTIZATION_RECIPE}, llm_model_type=function_gemma)...")
export_hf.export(
    model=BASE_MODEL_DIR,
    output_dir=BASE_OUTPUT_DIR,
    quantization_recipe=QUANTIZATION_RECIPE,
    litert_lm_model_type_override="function_gemma",
    moe_exports_implementation="eager",
)
print(f"✅ Base FunctionGemma model exported to: {BASE_LITERTLM_PATH} ({os.path.getsize(BASE_LITERTLM_PATH)/1e6:.2f} MB)")

### Step 8: Convert the adapter to `.tflite` (dynamic LoRA)

Packs the trained adapter into a `.tflite` file that LiteRT-LM can attach to a base model at runtime: `./functiongemma_270m_task_lora.tflite`. The companion notebook `FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb` attaches it to a base model exported with dynamic LoRA input slots.

In [ ]:
# @title { display-mode: "form" }
import os
import torch
import transformers
from litert_torch.generative.layers import lora as lora_utils

# Locate trained adapter directory
ADAPTER_DIR = "/content/function_gemma_lora" if os.path.exists("/content/function_gemma_lora") else "./functiongemma_270m_lora"
safetensors_path = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
if not os.path.exists(safetensors_path):
    raise FileNotFoundError(f"❌ Could not find adapter_model.safetensors in {ADAPTER_DIR}. Ensure training completed successfully.")

LORA_TFLITE_PATH = "./functiongemma_270m_task_lora.tflite"
print(f"📁 Ingesting trained LoRA weights from: {safetensors_path}")

# Map Hugging Face PEFT tensor names to LiteRT-LM conventions
tensor_names = lora_utils.LoRATensorNames(
    attn_query_w_a = "base_model.model.model.layers.{}.self_attn.q_proj.lora_A.weight",
    attn_query_w_b = "base_model.model.model.layers.{}.self_attn.q_proj.lora_B.weight",
    attn_key_w_a   = "base_model.model.model.layers.{}.self_attn.k_proj.lora_A.weight",
    attn_key_w_b   = "base_model.model.model.layers.{}.self_attn.k_proj.lora_B.weight",
    attn_value_w_a = "base_model.model.model.layers.{}.self_attn.v_proj.lora_A.weight",
    attn_value_w_b = "base_model.model.model.layers.{}.self_attn.v_proj.lora_B.weight",
    attn_output_w_a = "base_model.model.model.layers.{}.self_attn.o_proj.lora_A.weight",
    attn_output_w_b = "base_model.model.model.layers.{}.self_attn.o_proj.lora_B.weight",
)

config = transformers.AutoConfig.from_pretrained(BASE_MODEL_DIR)
if not hasattr(config, "num_layers"):
    config.num_layers = getattr(config, "num_hidden_layers", 18)

print("⚙️ Loading LoRA weights from safetensors...")
lora_obj = lora_utils.LoRA.from_safetensors(
    path = safetensors_path,
    scale = 2.0,
    config = config,
    lora_tensor_names = tensor_names,
)

# ==============================================================================
# 🎯 LiteRT-LM Memory-Mapped LoRA FlatBuffer Serializer
# ==============================================================================
# LiteRT-LM's LoraData (runtime/util/lora_data.cc)
# loads tensors via zero-copy mmap using: ReadData(buffer->offset(), buffer->size()).
# Raw tensor bytes are appended after the FlatBuffer header at 64KB alignment.
# ==============================================================================
def export_lora_to_litert_lm(lora_obj, output_path, base_offset=65536):
    """Serializes LoRA weights into the memory-mapped FlatBuffer format required by LiteRT-LM."""
    builder = lora_utils.flatbuffers.Builder(4096)
    schema_fb = lora_utils.schema_fb

    tensors, (names, _) = lora_utils._flatten_lora(lora_obj)
    names = [f"lora_{name}" for name in names]
    rank = lora_obj.get_rank()

    buffer_offsets = []
    # Buffer 0: empty buffer for convention
    schema_fb.BufferStart(builder)
    buffer_offsets.append(schema_fb.BufferEnd(builder))

    cur_offset = base_offset
    raw_bytes_list = []
    for tensor in tensors:
        t_bytes = tensor.detach().cpu().to(torch.float32).numpy().tobytes()
        raw_bytes_list.append(t_bytes)
        schema_fb.BufferStart(builder)
        schema_fb.BufferAddOffset(builder, cur_offset)
        schema_fb.BufferAddSize(builder, len(t_bytes))
        buffer_offsets.append(schema_fb.BufferEnd(builder))
        cur_offset += len(t_bytes)

    schema_fb.ModelStartBuffersVector(builder, len(buffer_offsets))
    for b_off in reversed(buffer_offsets):
        builder.PrependUOffsetTRelative(b_off)
    buffers_offset = builder.EndVector()

    tensor_offsets = []
    for i, (name, tensor) in enumerate(zip(names, tensors)):
        tensor_offsets.append(lora_utils._add_tensor(builder, name, tensor.shape, i + 1))

    schema_fb.SubGraphStartTensorsVector(builder, len(tensor_offsets))
    for t_off in reversed(tensor_offsets):
        builder.PrependUOffsetTRelative(t_off)
    tensors_offset = builder.EndVector()

    string_offset = builder.CreateString("lora_params")
    schema_fb.SubGraphStart(builder)
    schema_fb.SubGraphAddName(builder, string_offset)
    schema_fb.SubGraphAddTensors(builder, tensors_offset)
    subgraph_offset = schema_fb.SubGraphEnd(builder)

    schema_fb.ModelStartSubgraphsVector(builder, 1)
    builder.PrependUOffsetTRelative(subgraph_offset)
    subgraphs_offset = builder.EndVector()

    metadata_name_offset = builder.CreateString("lora_rank")
    schema_fb.MetadataStart(builder)
    schema_fb.MetadataAddName(builder, metadata_name_offset)
    schema_fb.MetadataAddBuffer(builder, rank)
    metadata_offset = schema_fb.MetadataEnd(builder)

    schema_fb.ModelStartMetadataVector(builder, 1)
    builder.PrependUOffsetTRelative(metadata_offset)
    metadata_vector_offset = builder.EndVector()

    model_desc = builder.CreateString("lora_params")
    schema_fb.ModelStart(builder)
    schema_fb.ModelAddVersion(builder, 3)
    schema_fb.ModelAddDescription(builder, model_desc)
    schema_fb.ModelAddBuffers(builder, buffers_offset)
    schema_fb.ModelAddSubgraphs(builder, subgraphs_offset)
    schema_fb.ModelAddMetadata(builder, metadata_vector_offset)
    model_offset = schema_fb.ModelEnd(builder)
    builder.Finish(model_offset, file_identifier=b"TFL3")
    flatbuffer_header = bytearray(builder.Output())

    if len(flatbuffer_header) > base_offset:
        raise ValueError(f"FlatBuffer header ({len(flatbuffer_header)} bytes) exceeds base_offset ({base_offset} bytes)")

    with open(output_path, "wb") as f:
        f.write(flatbuffer_header)
        f.write(b"\x00" * (base_offset - len(flatbuffer_header)))
        for raw_bytes in raw_bytes_list:
            f.write(raw_bytes)
    print(f"✅ Generated LiteRT-LM LoRA FlatBuffer: {output_path} ({os.path.getsize(output_path)/1e6:.2f} MB, rank={rank})")

print("⚙️ Packing LoRA weight matrices into 64 KB page-aligned TFLite FlatBuffer for LiteRT-LM...")
export_lora_to_litert_lm(lora_obj, LORA_TFLITE_PATH)

### Step 9: Merge the adapter into the base model (static fusion)

Merges the adapter into the base weights with PEFT's `merge_and_unload()`, then exports the result as a single `function_gemma` container: `/content/functiongemma_270m_fused_litertlm/model.litertlm`. This is the file you will run on your phone.

In [ ]:
# @title { display-mode: "form" }
import os
import torch
import transformers
from peft import PeftModel

FUSED_MODEL_DIR = "/content/functiongemma_270m_fused_hf"
FUSED_OUTPUT_DIR = "/content/functiongemma_270m_fused_litertlm"
FUSED_LITERTLM_PATH = os.path.join(FUSED_OUTPUT_DIR, "model.litertlm")

print("🔄 Merging the LoRA adapter into the base weights with PEFT merge_and_unload()...")
base_model = transformers.AutoModelForCausalLM.from_pretrained(BASE_MODEL_DIR, torch_dtype=torch.float32)
fused_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload()

print(f"💾 Saving fused PyTorch checkpoint to {FUSED_MODEL_DIR}...")
fused_model.save_pretrained(FUSED_MODEL_DIR)
transformers.AutoTokenizer.from_pretrained(BASE_MODEL_DIR).save_pretrained(FUSED_MODEL_DIR)
patch_fc_string_tools(FUSED_MODEL_DIR)

print(f"Exporting fused FunctionGemma 270M ({QUANTIZATION_RECIPE}, llm_model_type=function_gemma)...")
export_hf.export(
    model=FUSED_MODEL_DIR,
    output_dir=FUSED_OUTPUT_DIR,
    quantization_recipe=QUANTIZATION_RECIPE,
    litert_lm_model_type_override="function_gemma",
    moe_exports_implementation="eager",
)
print(f"✅ Fused single model created: {FUSED_LITERTLM_PATH} ({os.path.getsize(FUSED_LITERTLM_PATH)/1e6:.2f} MB)")

### Step 10: Evaluate the base and fine-tuned models

Passes the three smart-home tools to `engine.create_conversation(tools=...)` and runs the base model (Step 7) and the fused model (Step 9) with the LiteRT-LM runtime on the Colab CPU. It shows each model's call for the two example requests, then reports how often each model picks the right function and how often it produces the exact call (function **and** arguments) on held-out requests that never appear in training.

In [ ]:
import json
import os
import litert_lm

EVAL_N = 100  # @param {type:"integer"}
SFT_DATA_DIR = "/content/sft_data"
BASE_LITERTLM_PATH = globals().get("BASE_LITERTLM_PATH", "/content/functiongemma_270m_litertlm/model.litertlm")
FUSED_LITERTLM_PATH = globals().get("FUSED_LITERTLM_PATH", "/content/functiongemma_270m_fused_litertlm/model.litertlm")

with open(f"{SFT_DATA_DIR}/tools.json", encoding="utf-8") as f:
    TOOLS = json.load(f)
with open(f"{SFT_DATA_DIR}/eval_toolcalls.jsonl", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f][:EVAL_N]


class SmartHomeTool(litert_lm.Tool):
    """Passes the exact OpenAPI schema used for training to LiteRT-LM."""

    def __init__(self, definition):
        self._definition = definition

    def get_tool_description(self):
        return {"type": "function", "function": self._definition}

    def execute(self, param):
        return "ok"


def first_tool_call(reply):
    calls = reply.get("tool_calls") if isinstance(reply, dict) else getattr(reply, "tool_calls", None)
    if not calls:
        return None, None
    call = calls[0]
    if not isinstance(call, dict):  # litert_lm.ToolCall-style object
        return getattr(call, "name", None), dict(getattr(call, "arguments", {}) or {})
    fn = call.get("function", call)
    args = fn.get("arguments", {})
    if isinstance(args, str):
        args = json.loads(args or "{}")
    return fn.get("name"), dict(args)


def norm_args(args):
    return {k: str(v).strip().lower() for k, v in (args or {}).items()}


cpu_backend = litert_lm.Backend.CPU() if callable(getattr(litert_lm.Backend, "CPU", None)) else litert_lm.Backend.CPU
tools = [SmartHomeTool(t) for t in TOOLS]
DEMO_EXAMPLES = eval_rows[:2]  # "Turn on the living room lights", "I want the office at 22 degrees"
results, demo_calls = {}, {}

for label, model_path in [("Base", BASE_LITERTLM_PATH), ("Fine-tuned", FUSED_LITERTLM_PATH)]:
    name_ok = exact_ok = 0
    demo_calls[label] = []
    with litert_lm.Engine(model_path=model_path, backend=cpu_backend) as engine:
        for i, row in enumerate(eval_rows):
            with engine.create_conversation(tools=tools, automatic_tool_calling=False) as conv:
                reply = conv.send_message(row["query"], max_output_tokens=64)
            name, args = first_tool_call(reply)
            correct = name == row["name"] and norm_args(args) == norm_args(row["arguments"])
            if i < len(DEMO_EXAMPLES):
                demo_calls[label].append((name, args, correct))
            name_ok += name == row["name"]
            exact_ok += correct
    results[label] = (name_ok / len(eval_rows), exact_ok / len(eval_rows))

print("#" * 80 + "\n📊 SIDE-BY-SIDE (SxS) VERIFICATION SUMMARY\n" + "#" * 80)
for i, row in enumerate(DEMO_EXAMPLES):
    print("=" * 80 + f"\nPROMPT:   {row['query']}\nEXPECTED: {row['name']}({json.dumps(row['arguments'])})")
    for label in ("Base", "Fine-tuned"):
        name, args, correct = demo_calls[label][i]
        shown = f"{name}({json.dumps(args)})" if name else "(no tool call)"
        print(f"  {'✅' if correct else '❌'} {label:<11} {shown}")

print(f"\nHeld-out smart-home requests (n={len(eval_rows)}, unseen phrasings/devices):")
print(f"{'Model':<12}{'Correct function':>18}{'Exact call':>14}")
for label, (fn_acc, exact_acc) in results.items():
    print(f"{label:<12}{fn_acc:>18.1%}{exact_acc:>14.1%}")

## Run it on your phone

### Step 11: Publish to Hugging Face and get a QR code

Uploads the models to an **individual directory** (`HF_EXPORT_SUBFOLDER`) inside your Hugging Face repository (`HF_EXPORT_REPO_ID`), so multiple runs or users never overwrite shared files. Check `UPLOAD_TO_HF_HUB` and set `HF_EXPORT_REPO_ID` to a repository you can write to.

| File in the subfolder | What it is | Used by |
|---|---|---|
| `functiongemma_270m_fused_litertlm/model.litertlm` | Fused model (Step 9) | AI Edge Gallery (Step 12), companion notebook |
| `functiongemma_270m_base_litertlm/model.litertlm` | Base model (Step 7) | Comparisons |
| `functiongemma_270m_task_lora.tflite` | Adapter (Step 8) | Dynamic LoRA |
| `functiongemma_270m_lora/adapter_model.safetensors` | Raw PEFT adapter (Step 4) | Companion notebook (dynamic LoRA) |
| `tools.json`, `gallery_system_prompt.txt` | Tool schema and Gallery system prompt (Step 3) | Companion notebook, Step 12 |

After uploading, the cell prints a QR code that links to the **fused** model. It always prints the AI Edge Gallery system prompt for Step 12.

In [ ]:
# @title { display-mode: "form" }
import io
import os
import qrcode
from IPython.display import Image as IPyImage, display

UPLOAD_TO_HF_HUB = False  # @param {type:"boolean"}
HF_EXPORT_REPO_ID = "your-hf-username/functiongemma-270m-lora-litertlm"  # @param {type:"string"}
HF_EXPORT_SUBFOLDER = "functiongemma_270m_lora_gpu"  # @param {type:"string"}

subfolder = HF_EXPORT_SUBFOLDER.strip("/") or "functiongemma_270m_lora_gpu"
# Repo layout inside the subfolder. FunctionGemma_270M_2_LoRA_on_LiteRT_LM.ipynb downloads these
# exact names, so keep both notebooks in sync if you rename anything.
uploads = {
    f"{subfolder}/functiongemma_270m_fused_litertlm/model.litertlm": globals().get(
        "FUSED_LITERTLM_PATH", "/content/functiongemma_270m_fused_litertlm/model.litertlm"),
    f"{subfolder}/functiongemma_270m_base_litertlm/model.litertlm": globals().get(
        "BASE_LITERTLM_PATH", "/content/functiongemma_270m_litertlm/model.litertlm"),
    f"{subfolder}/functiongemma_270m_task_lora.tflite": globals().get(
        "LORA_TFLITE_PATH", "/content/functiongemma_270m_task_lora.tflite"),
    f"{subfolder}/functiongemma_270m_lora/adapter_model.safetensors": "/content/functiongemma_270m_lora/adapter_model.safetensors",
    f"{subfolder}/tools.json": "/content/sft_data/tools.json",
    f"{subfolder}/gallery_system_prompt.txt": "/content/sft_data/gallery_system_prompt.txt",
}
fused_path_in_repo = next(iter(uploads))

if UPLOAD_TO_HF_HUB:
    if HF_EXPORT_REPO_ID.startswith("your-hf-username/"):
        raise ValueError("Please set HF_EXPORT_REPO_ID to your own Hugging Face repository (e.g. 'username/functiongemma-270m-lora-litertlm').")
    missing = [p for p in uploads.values() if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(f"Cannot upload, files not found: {missing}. Run Steps 3-9 first.")
    from huggingface_hub import HfApi
    api = HfApi(token=globals().get("hf_token"))
    api.create_repo(repo_id=HF_EXPORT_REPO_ID, repo_type="model", exist_ok=True)
    for path_in_repo, local_path in uploads.items():
        api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo, repo_id=HF_EXPORT_REPO_ID, repo_type="model")
        print(f"[OK] Uploaded {path_in_repo} to https://huggingface.co/{HF_EXPORT_REPO_ID}")

    download_url = f"https://huggingface.co/{HF_EXPORT_REPO_ID}/resolve/main/{fused_path_in_repo}"
    print(f"\nDirect fused .litertlm URL: {download_url}")
    print("Scan the QR code with your phone to download the fused model, then import it in AI Edge Gallery (Step 12):")
    qr_img = qrcode.make(download_url)
    buf = io.BytesIO()
    qr_img.save(buf, format="PNG")
    display(IPyImage(data=buf.getvalue(), width=220))
else:
    print("Skipped Hugging Face upload. Check UPLOAD_TO_HF_HUB and set HF_EXPORT_REPO_ID to upload and get a download URL + QR code.")

with open("/content/sft_data/gallery_system_prompt.txt", encoding="utf-8") as f:
    print("\nAI Edge Gallery system prompt (Step 12):\n" + f.read())

### Step 12: Run the model in AI Edge Gallery

The QR code from Step 11 links to the fused `model.litertlm` in your Hugging Face repo. Scan it with your phone to download the file, then load it into the [Google AI Edge Gallery](https://developers.google.com/edge/gallery) app.

1. **Get the model onto the phone.** Scan the QR code above with your phone's camera and save `model.litertlm` to the device.

2. **Import it.** Open AI Edge Gallery and tap the floating **Import model** button. Choose **from local file** and pick the downloaded `model.litertlm`. (Or skip the download: tap **from Hugging Face** and paste the fused model's URL. Gallery resolves `https://huggingface.co/<user>/<repo>/resolve/main/<path>.litertlm` itself.) Only `.litertlm` and `.task` files are accepted.

3. **Set the system prompt.** Open the imported model, tap the settings icon in the top bar, switch to the **System prompt** tab, and paste the system prompt printed at the end of Step 11 (also saved as `gallery_system_prompt.txt`). It contains the three tool declarations, so the phone sees the same prompt format as training. Tap **OK**.

4. **Ask it something.** Open **AI Chat** with the model selected and send one of the example requests:

    ```
    Turn on the living room lights
    ```

    ```
    I want the office at 22 degrees
    ```

    The fine-tuned model answers with a function call such as `call:set_device_state{device_id:light.living_room,state:on}`.

> [!NOTE]
> The QR code links to the fused model only. AI Edge Gallery cannot attach a LoRA adapter at runtime, so the fused model is the only file that shows the fine-tuned behavior in the app.

## Next Steps: Run Your `.litertlm` Model Everywhere

Once you have exported your `.litertlm` model, you can deploy and test it across mobile, web, and desktop:

- **[LiteRT-LM CLI (`litert-lm`)](https://developers.google.com/edge/litert-lm/cli)**: Run, benchmark, and serve `.litertlm` models from your terminal (`pip install litert-lm && litert-lm run model.litertlm`).
- **[OpenAI-Compatible Server](https://developers.google.com/edge/litert-lm/cli/openai_server#sending_chat_completion_requests)**: Serve the model with `litert-lm serve` and send it chat-completion requests, including tools.
- **[LiteRT-LM Web Demo](https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html)**: Upload your `.litertlm` package in Chrome or Edge to run 100% client-side in the browser with WebGPU acceleration.
- **[Google AI Edge Gallery App](https://developers.google.com/edge/gallery)**: Import your `.litertlm` model on Android or iOS (via your Hugging Face download URL or QR code) for interactive on-device inference.